# Image Super-Resolution

A neural field can upsample an image: fitted to the low-resolution (LR) pixels
through the degradation that made them, it renders the image at high resolution
(HR). This notebook recovers image 0882 of the DIV2K validation set from its 4×
bicubic downsampling under the super-resolution benchmark's protocol: about 500k
parameters each, and 1000 steps of Adam, each on 135 random blocks of 8 × 8 LR
pixels, with a cosine-annealed learning rate. It compares FUTON, with a sinc and
a cosine basis, against four representative baselines of the benchmark: SIREN
and FINER (periodic activations), Instant-NGP (a hash grid) and TensoRF (a grid
factorization), with bicubic interpolation as the reference. No model sees the
HR image, which only scores the results. Every setting is that of
`configs/super_resolution.yaml`, which holds every model of the benchmark.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
from matplotlib.ticker import (
    FixedLocator,
    MaxNLocator,
    MultipleLocator,
    NullFormatter,
    StrMethodFormatter,
)
from matplotlib_inline.backend_inline import set_matplotlib_formats
from PIL import Image
from torchvision.transforms.functional import pil_to_tensor

import neurofield as nf

set_matplotlib_formats("retina")  # sharp figures on high-resolution screens
sns.set_theme(
    context="notebook", style="whitegrid", font_scale=0.85, rc={"grid.color": ".92"}
)

colorblind = sns.color_palette("colorblind")
# FUTON in black, its cosine basis dashed; the periodic activations in warm
# hues and the grids in cool ones.
PALETTE = {
    "FUTON (sinc)": "black",
    "FUTON (cosine)": "black",
    "RFF": colorblind[4],
    "SIREN": colorblind[1],
    "FINER": colorblind[3],
    "Instant-NGP": colorblind[2],
    "TensoRF": colorblind[0],
}
DASHES = {model: (4, 2) if model == "FUTON (cosine)" else "" for model in PALETTE}
# seaborn's own markers, each fixed to a model as its colour is
MARKERS = dict(zip(PALETTE, ["o", "X", (4, 0, 45), "P", (4, 0, 0), (4, 1, 0), "^"]))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

The plotting helpers: training time on a log axis, the metric on whole
decibels or half points, a summary table, images side by side, and
reconstructions above their errors.

In [ ]:
NICE = [1, 2, 5, 10]  # tick steps: 1, 2 or 5 times a power of ten


def time_axis(ax, times):
    """Training time on a log axis, ticked 1, 2, 5, 10, ... in plain numbers."""
    low, high = times.min() / 1.1, times.max() * 1.1
    ax.set(xscale="log", xlim=(low, high), xlabel="Training time (s)")
    for subs in ((1, 2, 5), (1,)):
        ticks = [m * 10.0**k for k in range(-3, 6) for m in subs]
        ticks = [tick for tick in ticks if low <= tick <= high]
        if len(ticks) <= 7:
            break
    ax.xaxis.set_major_locator(FixedLocator(ticks))
    ax.xaxis.set_major_formatter(StrMethodFormatter("{x:g}"))
    ax.xaxis.set_minor_formatter(NullFormatter())


def metric_axis(ax, metric, low, high):
    """The metric's axis over [low, high]: whole decibels for a PSNR, half
    points for an IoU, which cannot pass 100."""
    if metric == "IoU (%)":
        low = np.floor(2 * low) / 2
        ax.set_ylim(low, min(max(np.ceil(2 * high) / 2, low + 0.5), 100))
        ax.yaxis.set_major_locator(MultipleLocator(0.5))
    else:
        low = np.floor(low)
        ax.set_ylim(low, max(np.ceil(high), low + 1))
        ax.yaxis.set_major_locator(MaxNLocator(nbins=6, steps=NICE, integer=True))


def table(summary, formats, better):
    """The summary, each column in its format and, in each column of
    ``better`` ("higher" or "lower"), the best value bold and the second
    underlined."""

    def marks(column):
        rank = column.rank(method="min", ascending=better[column.name] == "lower")
        looks = {1: "font-weight: bold", 2: "text-decoration: underline"}
        return [looks.get(r, "") for r in rank]

    return (
        summary.style.format(formats, na_rep="")
        .apply(marks, subset=list(better))
        .hide(axis="index")
    )


def show(panels, ncols, width=11, **imshow):
    """Images side by side under their titles, in rows of ``ncols``, each
    drawn at the size of the first."""
    height, side = panels[0][1].shape[:2]
    nrows = -(-len(panels) // ncols)
    room = 0.45 if any("\n" in title for title, _ in panels) else 0.3
    fig, axes = plt.subplots(
        nrows,
        ncols,
        squeeze=False,
        layout="constrained",
        figsize=(width, nrows * (width / ncols * height / side + room)),
    )
    for ax in axes.flat:
        ax.axis("off")
    for ax, (title, image) in zip(axes.flat, panels):
        ax.imshow(image, interpolation="nearest", **imshow)
        ax.set_title(title, fontsize=9)
    return fig


def compare(panels, label, width=11):
    """The reference and the reconstructions in a row under their titles
    (``panels``, the reference first), and beneath each reconstruction its
    absolute error, averaged over the colour channels; the errors share one
    scale, which saturates at their 99th percentile, and its colour bar
    stands beneath the reference."""
    reference = panels[0][1].astype(float)
    errors = [np.abs(image - reference).mean(-1) for _, image in panels[1:]]
    height, side = reference.shape[:2]
    ncols = len(panels)
    fig, axes = plt.subplots(
        2,
        ncols,
        layout="constrained",
        figsize=(width, 2 * width / ncols * height / side + 0.45),
    )
    for ax in axes.flat:
        ax.axis("off")
    for ax, (title, image) in zip(axes[0], panels):
        ax.imshow(image, interpolation="nearest")
        ax.set_title(title, fontsize=9)
    vmax = np.percentile(np.concatenate([error.ravel() for error in errors]), 99)
    for ax, error in zip(axes[1, 1:], errors):
        maps = ax.imshow(
            error, cmap="magma", vmin=0, vmax=vmax, interpolation="nearest"
        )
    bar = fig.colorbar(maps, cax=axes[1, 0].inset_axes([0.8, 0.04, 0.07, 0.92]))
    bar.ax.yaxis.set_ticks_position("left")
    bar.ax.yaxis.set_label_position("left")
    bar.ax.yaxis.set_major_locator(MaxNLocator(nbins=4, steps=NICE))
    bar.ax.tick_params(labelsize=8)
    bar.set_label(label, fontsize=9)
    bar.outline.set_visible(False)
    return fig

## Data

The HR image and the LR image that the fields see, with the region that the
comparison at the end magnifies: the butterfly's head, whose antennae are
thinner than an LR pixel.

In [ ]:
SCALE = 4
HR_PATH = "../data/DIV2K/DIV2K_valid_HR/0882.png"
LR_PATH = f"../data/DIV2K/DIV2K_valid_LR_bicubic/X{SCALE}/0882x{SCALE}.png"
CROP = (slice(380, 636), slice(960, 1216))  # rows and columns: the butterfly's head

image = nf.ImageCoordinateDataset(HR_PATH)  # every HR pixel, for evaluation
image_lr = Image.open(LR_PATH).convert("RGB")
(H, W), (h, w) = image.grid_size, (image_lr.height, image_lr.width)
original = image.original.permute(1, 2, 0).numpy()  # (H, W, 3), uint8
print(f"HR: {H} x {W} pixels, LR: {h} x {w}")

fig = show(
    [
        ("High resolution", original),
        (f"Low resolution ({SCALE}×)", np.asarray(image_lr)),
    ],
    ncols=2,
)
for ax, scale in zip(fig.axes, (1, SCALE)):  # the region on either image
    rows, columns = (slice(s.start // scale, s.stop // scale) for s in CROP)
    box = plt.Rectangle(
        (columns.start, rows.start),
        columns.stop - columns.start,
        rows.stop - rows.start,
    )
    ax.add_patch(box).set(fill=False, edgecolor="red", linewidth=1.5)
plt.show()

## Bicubic Interpolation

Every result is scored against the HR image as is standard: PSNR and SSIM on
luma, and LPIPS on RGB, all without a 4-pixel border. Bicubic interpolation,
the standard reference, needs no training; PIL's bicubic filter is that of
MATLAB's `imresize`. The plots show its PSNR as a dotted level.

In [ ]:
def shave(image):
    """A uint8 RGB image ``(3, H, W)`` on the device, without its border."""
    return image.to(device)[:, SCALE:-SCALE, SCALE:-SCALE]


def luma(image):
    """The BT.601 luma of the shaved image, as MATLAB's rgb2ycbcr computes it."""
    weights = torch.tensor([65.481, 128.553, 24.966], device=device) / 255
    return 16 + torch.einsum("chw,c->hw", shave(image).float(), weights)[None]


METRICS = {  # each called as metric(reconstruction, original)
    "psnr": lambda pred, target: nf.psnr(luma(pred), luma(target), data_range=255),
    "ssim": lambda pred, target: nf.ssim(luma(pred), luma(target), data_range=255),
    "lpips": lambda pred, target: nf.lpips(shave(pred), shave(target)),
}

upsampled = image_lr.resize((W, H), Image.Resampling.BICUBIC)
bicubic = {
    name: metric(pil_to_tensor(upsampled), image.original)
    for name, metric in METRICS.items()
}
print(f"Bicubic: {bicubic['psnr']:.2f} dB")


def bicubic_level(ax, labelled=False):
    """Bicubic interpolation's PSNR as a dotted level, labelled at the right."""
    ax.axhline(bicubic["psnr"], color=".3", linestyle=":", linewidth=1.2)
    if labelled:
        ax.annotate(
            "Bicubic",
            (1, bicubic["psnr"]),
            xycoords=("axes fraction", "data"),
            xytext=(-3, 3),
            textcoords="offset points",
            ha="right",
            va="bottom",
            fontsize=8,
            color=".3",
        )

## Forward Model

The benchmark's degradation, antialiased bicubic downsampling as MATLAB's
`imresize` computes it, is linear and separable: one matrix per axis describes
it exactly, borders included, and resizing an identity image along that axis
writes the matrix out. Each of its rows holds at most `4 * SCALE` nonzero taps,
the footprint of one LR pixel along the axis. Applied to the HR image, the
forward model gives back the benchmark's LR image, up to rounding.

In [ ]:
def resize_matrix(size_in, size_out):
    """The ``(size_out, size_in)`` matrix of antialiased bicubic resizing."""
    identity = torch.eye(size_in, device=device)[None, None]
    size = (size_out, size_in)
    return F.interpolate(identity, size=size, mode="bicubic", antialias=True)[0, 0]


def downsample(pixels, kernels):
    """Apply one kernel per axis: ``(..., H, W, C)`` to ``(..., h, w, C)``."""
    return torch.einsum("...il,...lmc,...jm->...ijc", kernels[0], pixels, kernels[1])


kernels = [resize_matrix(H, h), resize_matrix(W, w)]  # (h, H) and (w, W)
print(f"Taps per LR pixel and axis: {int((kernels[0] != 0).sum(1).max())}")

simulated = downsample(image.original.movedim(0, -1).float().to(device), kernels)
simulated = simulated.round().clamp(0, 255).movedim(-1, 0).cpu()
fidelity = nf.psnr(simulated, pil_to_tensor(image_lr), data_range=255)
print(f"Forward model against the LR image: {fidelity:.2f} dB")

## Minibatches

Each step fits a fixed number of random blocks of LR pixels, whatever the
image's size, and the field renders only the HR pixels under their kernels. Neighbouring LR pixels share
most of their footprint, so a block needs far fewer field evaluations per pixel
than single pixels do: 30 in an 8 × 8 block against 256 alone. The loss
downsamples each rendered footprint along both axes and takes the mean absolute
error against the LR pixels, the usual loss in super-resolution.

In [ ]:
class SuperResolutionDataset(nf.ImageCoordinateDataset):
    """LR pixels as targets, the HR coordinates under their kernels as inputs.

    Each item draws ``num_blocks`` (B) random square blocks of ``block_size``
    (S) LR pixels a side, whose footprints span L HR pixels a side. It holds
    ``input``, the HR coordinates of each footprint, of shape ``(B, L, L, 2)``;
    ``weight_y`` and ``weight_x``, the kernel rows of each block on its
    footprint, of shape ``(B, S, L)``; and ``target``, the LR pixels in
    ``[-1, 1]``, of shape ``(B, S, S, 3)``.
    """

    def __init__(self, image, kernels, num_blocks, block_size=8):
        self.kernels = kernels
        self.num_blocks, self.block_size = num_blocks, block_size
        self.grid_size = tuple(kernel.shape[1] for kernel in kernels)  # (H, W)
        device = kernels[0].device
        self.input = self.create_input().to(device)  # HR coordinates, (H, W, 2)
        self.target = self.preprocess(self.load(image, mode="RGB")).to(device)
        # The first tap of each kernel row, and the side L of a block's
        # footprint: one row's taps, and scale more per further row.
        self.first = [(kernel != 0).int().argmax(1) for kernel in kernels]
        taps = max(int((kernel != 0).sum(1).max()) for kernel in kernels)
        scale = self.grid_size[0] // self.target.shape[0]
        self.footprint = scale * (block_size - 1) + taps

    def __getitem__(self, index):
        (rows, taps_y, weight_y), (columns, taps_x, weight_x) = map(self.draw, (0, 1))
        return {
            "input": self.input[taps_y[:, :, None], taps_x[:, None, :]],
            "weight_y": weight_y,
            "weight_x": weight_x,
            "target": self.target[rows[:, :, None], columns[:, None, :]],
        }

    def draw(self, axis):
        """Random blocks along an axis: their LR pixels, the HR pixels of their
        footprints, and the kernel rows on those."""
        kernel, first = self.kernels[axis], self.first[axis]
        (size, full), device = kernel.shape, kernel.device
        start = torch.randint(
            size - self.block_size + 1, (self.num_blocks, 1), device=device
        )
        pixels = start + torch.arange(self.block_size, device=device)  # (B, S)
        begin = first[start].clamp(max=full - self.footprint)  # inside the image
        taps = begin + torch.arange(self.footprint, device=device)  # (B, L)
        return pixels, taps, kernel[pixels[:, :, None], taps[:, None, :]]


def field_loss(batch, model):
    """Render each footprint, downsample it along both axes, and match the LR
    pixels: the mean absolute error, with the fit's PSNR for the training log."""
    weights = batch["weight_y"], batch["weight_x"]
    output = downsample(model(batch["input"]), weights)
    loss = F.l1_loss(output, batch["target"])
    mse = F.mse_loss(output.detach(), batch["target"])
    return loss, {"l1": loss.detach(), "psnr": 10 * torch.log10(4 / mse)}

## Models

Each entry gives a model's class, its arguments and its learning rate.
`# deviation` marks a setting that departs from the authors' code, with the
reason. Every learning rate, and every setting marked `# tuned`, maximizes the
mean PSNR over three DIV2K training images (0395, 0431 and 0777) under this
protocol.

Every model has about 500k parameters, close to the 519k values of the LR
image, so that each can fit what it observes and the comparison rests on what
it adds between the pixels. A baseline takes the smallest size of its kind that
reaches the budget, and FUTON the largest CP rank within it, so FUTON is never
the larger model. FUTON has one basis function per LR pixel along each axis, the
bandwidth that the LR image determines.

In [ ]:
# FUTON's rank: the largest within 500k parameters, where rank R costs R * (h + w)
# factor entries and R * 128 + 128 in the hidden layer of the decoder, whose output
# layer adds 3 * 128 + 3.
RANK = (500_000 - 4 * 64 - 3) // (h + w + 64)

CONFIGS = [
    {
        "name": "SIREN",  # FINER image benchmark
        "class": nf.SIREN,
        "kwargs": {
            "hidden_features": 407,
            "hidden_layers": 4,
            "omega": 120.0,  # tuned; 60 loses 0.7 dB and 240 diverges
            "output_activation": torch.tanh,
        },
        "lr": 1e-4,  # every higher rate diverges
    },
    {
        "name": "FINER",  # FINER image benchmark
        "class": nf.FINER,
        "kwargs": {
            "hidden_features": 407,
            "hidden_layers": 4,
            "omega": 120.0,  # tuned; 30 and 60 are worse and 240 diverges
            "output_activation": torch.tanh,
        },
        "lr": 1e-4,  # every higher rate diverges
    },
    {
        "name": "Instant-NGP",  # instant-ngp configs/image/base.json
        "class": nf.InstantNGP,
        "kwargs": {
            "num_levels": 17,  # deviation: 16 in the reference; 17 meets the budget
            "features_per_level": 2,
            "log2_hashmap_size": 15,
            "base_resolution": 16,
            "max_resolution": max(h, w),  # tuned: the LR grid
            "hidden_features": 64,
            "hidden_layers": 2,
            "output_activation": torch.tanh,
        },
        # PyTorch's default Adam, as every model; with Instant-NGP's own betas
        # (0.9, 0.99) and eps 1e-15, 1e-2 lost 0.06 dB
        "lr": 3e-2,
    },
    {
        "name": "TensoRF",
        "class": nf.TensoRF,
        "kwargs": {
            "rank": 566,
            "resolution": max(h, w) // 2,  # tuned: half the LR grid
            "mode": "cp",
            "hidden_features": 256,
            "hidden_layers": 2,
            "output_activation": torch.tanh,
        },
        "lr": 3e-2,  # tuned; 1e-2 loses 0.1 dB
    },
    {
        "name": "FUTON (sinc)",
        "class": nf.FUTON,
        "kwargs": {
            "basis": (  # on the HR pixels
                "sinc",
                {"num_components": (h, w), "grid_size": (H, W)},
            ),
            "combiner": ("cp", {"rank": RANK}),
            "decoder": ("mlp", {"hidden_layers": 1, "hidden_features": 64}),
            "output_activation": torch.tanh,
        },
        "lr": 1e-2,  # 3e-2 costs 1.3 dB on three images, 3e-3 1.2
    },
    {
        "name": "FUTON (cosine)",
        "class": nf.FUTON,
        "kwargs": {
            "basis": (  # on the HR pixels
                "cosine",
                {"num_components": (h, w), "grid_size": (H, W)},
            ),
            "combiner": ("cp", {"rank": RANK}),
            "decoder": ("mlp", {"hidden_layers": 1, "hidden_features": 64}),
            "output_activation": torch.tanh,
        },
        "lr": 1e-2,  # 3e-3 costs 0.6 dB on three images, 3e-2 4.5
    },
]

## Training

Each step fits 135 blocks of 8 × 8 LR pixels, whose footprints hold 261,360 HR
pixels, about 2^18. Every 50 steps the field renders the whole HR image, which
is scored.

Every model is compiled with `torch.compile`, and an untimed forward and
backward pass compiles it before training starts. The training times exclude
that compilation and the evaluations.

In [ ]:
CHUNK = 2**18  # HR pixels per forward pass when evaluating
train_dataset = SuperResolutionDataset(LR_PATH, kernels, num_blocks=135, block_size=8)
results = {}

for config in CONFIGS:
    torch.manual_seed(0)
    model = config["class"](in_features=2, out_features=3, **config["kwargs"])
    print(f"{config['name']}: {nf.count_parameters(model):,} parameters")
    model.to(device).compile()
    model(train_dataset[0]["input"][None].to(device)).sum().backward()  # compiles it
    model.zero_grad(set_to_none=True)
    result = nf.train(
        model,
        train_dataset,
        image,
        loss_fn=field_loss,
        lr=config["lr"],
        num_steps=1000,
        metrics=METRICS,
        log_interval=50,
        eval_interval=50,
        chunk_size=CHUNK,
        device=device,
        **config.get("train_kwargs", {}),
    )
    output = nf.chunked_inference(model, image.input, chunk_size=CHUNK, device=device)
    result["reconstruction"] = image.postprocess(output.cpu()).permute(1, 2, 0).numpy()
    results[config["name"]] = result

    del model, output, result["model_dict"]
    torch.cuda.empty_cache()
    torch.compiler.reset()  # the next model compiles anew

## Convergence

The PSNR of the HR image, every 50 steps; the dotted level is bicubic
interpolation's.

FUTON is black, its cosine basis dashed; the periodic activations have warm
hues and the grids cool ones.

In [ ]:
records = pd.DataFrame(
    {
        "Model": name,
        "Step": entry["step"],
        "Time (s)": entry["elapsed"],
        "PSNR (dB)": entry["eval"]["psnr"],
    }
    for name, result in results.items()
    for entry in result["history"]
    if "eval" in entry
)

order = [model for model in PALETTE if model in set(records["Model"])]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True, layout="constrained")
for ax, x in zip(axes, ("Step", "Time (s)")):
    sns.lineplot(
        records,
        x=x,
        y="PSNR (dB)",
        hue="Model",
        style="Model",
        hue_order=order,
        style_order=order,
        palette=PALETTE,
        dashes=DASHES,
        legend=ax is axes[1],
        ax=ax,
    )
    bicubic_level(ax, labelled=ax is axes[0])
axes[0].xaxis.set_major_locator(MaxNLocator(nbins=5, steps=NICE))
axes[0].xaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
final = records.groupby("Model")["PSNR (dB)"].last()
low = min(final.min(), bicubic["psnr"])
high = max(records["PSNR (dB)"].max(), bicubic["psnr"])
margin = 0.05 * (high - low)  # so that no curve ends on the frame
metric_axis(axes[0], "PSNR (dB)", low - margin, high + margin)
shown = records["PSNR (dB)"] >= axes[0].get_ylim()[0]  # the evaluations in view
time_axis(axes[1], records.loc[shown, "Time (s)"])

sns.move_legend(
    axes[-1], "upper left", bbox_to_anchor=(1.02, 1), title=None, frameon=False
)
sns.despine(fig, offset=3)
plt.show()

## Summary

Throughput is the median rate of the evaluations, which reconstruct the whole
HR image, and inference memory their median peak; training memory is the peak
of training. Bicubic interpolation trains nothing and runs on the CPU, so it
has scores but no costs. In each column the best value is bold and the second
underlined.

In [ ]:
rows = []
for name, result in results.items():
    evaluations = [entry["eval"] for entry in result["history"] if "eval" in entry]
    duration = np.median([e["duration"] for e in evaluations])
    memory = np.median([e["peak_memory"] for e in evaluations])
    rows.append(
        {
            "Model": name,
            "Params (k)": result["config"]["num_params"] / 1e3,
            "Training time (s)": result["history"][-1]["elapsed"],
            "Training memory (MB)": result["config"]["peak_memory"] / 1e6,
            "Throughput (MPix/s)": H * W / duration / 1e6,
            "Inference memory (MB)": memory / 1e6,
            "PSNR (dB)": evaluations[-1]["psnr"],
            "SSIM": evaluations[-1]["ssim"],
            "LPIPS": evaluations[-1]["lpips"],
        }
    )
rows.append(
    {
        "Model": "Bicubic",
        "PSNR (dB)": bicubic["psnr"],
        "SSIM": bicubic["ssim"],
        "LPIPS": bicubic["lpips"],
    }
)
summary = pd.DataFrame(rows).sort_values("PSNR (dB)", ascending=False)

formats = {
    "Params (k)": "{:.1f}",
    "Training time (s)": "{:.1f}",
    "Training memory (MB)": "{:.0f}",
    "Throughput (MPix/s)": "{:.1f}",
    "Inference memory (MB)": "{:.0f}",
    "PSNR (dB)": "{:.2f}",
    "SSIM": "{:.3f}",
    "LPIPS": "{:.3f}",
}
better = {
    "Training time (s)": "lower",
    "Training memory (MB)": "lower",
    "Throughput (MPix/s)": "higher",
    "Inference memory (MB)": "lower",
    "PSNR (dB)": "higher",
    "SSIM": "higher",
    "LPIPS": "lower",
}
table(summary, formats, better)

In [ ]:
fields = summary[summary["Model"].isin(results)]  # the trained models

order = [model for model in PALETTE if model in set(fields["Model"])]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True, layout="constrained")
for ax, cost in zip(axes, ("Training time (s)", "Throughput (MPix/s)")):
    sns.scatterplot(
        fields,
        x=cost,
        y="PSNR (dB)",
        hue="Model",
        style="Model",
        hue_order=order,
        style_order=order,
        palette=PALETTE,
        markers=MARKERS,
        s=70,
        legend=ax is axes[1],
        ax=ax,
    )
    bicubic_level(ax, labelled=ax is axes[0])
    ax.margins(x=0.08)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5, steps=NICE))
axes[0].set_xlabel("Training time (s) ↓")
axes[1].set_xlabel("Throughput (MPix/s) ↑")
low = min(fields["PSNR (dB)"].min(), bicubic["psnr"])
high = max(fields["PSNR (dB)"].max(), bicubic["psnr"])
metric_axis(axes[0], "PSNR (dB)", low - 0.08 * (high - low), high + 0.08 * (high - low))

sns.move_legend(
    axes[-1], "upper left", bbox_to_anchor=(1.02, 1), title=None, frameon=False
)
sns.despine(fig, offset=3)
plt.show()

## Qualitative Comparison

The butterfly's head in every reconstruction, titled with its PSNR on the whole
image, and beneath, the error of each, averaged over the colour channels, on
one scale. The antennae are thinner than an LR pixel, so what a model draws
there comes from its inductive bias.

In [ ]:
reconstructions = {name: result["reconstruction"] for name, result in results.items()}
reconstructions["Bicubic"] = np.asarray(upsampled)
crops = [("Original", original[CROP])] + [
    (f"{name}\n{psnr:.2f} dB", reconstructions[name][CROP])
    for name, psnr in zip(summary["Model"], summary["PSNR (dB)"])
]
compare(crops, "Error (8-bit levels)")
plt.show()